In [13]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

# load master df saved from EDA
df = pd.read_csv('../data/processed/master_df.csv')

print("Shape:", df.shape)
print("Columns:\n", df.columns.tolist())
print("\nNulls:", df.isnull().sum().sum())
print("\nTarget distribution:")
print(df['churn_flag'].value_counts())
print(df['churn_flag'].value_counts(normalize=True).round(3))

Shape: (500, 36)
Columns:
 ['account_id', 'account_name', 'industry', 'country', 'signup_date', 'referral_source', 'plan_tier', 'seats', 'is_trial', 'churn_flag', 'max_mrr', 'mean_mrr', 'total_seats_sub', 'sub_count', 'ever_upgraded', 'ever_downgraded', 'auto_renew', 'billing_frequency', 'plan_tier_sub', 'ticket_count', 'mean_resolution_hrs', 'mean_first_response', 'mean_satisfaction', 'escalation_count', 'total_usage_count', 'total_usage_duration', 'total_errors', 'distinct_features', 'beta_feature_usage', 'churn_event_count', 'total_refund', 'ever_reactivated', 'preceding_downgrade', 'preceding_upgrade', 'most_recent_reason', 'has_satisfaction_score']

Nulls: 0

Target distribution:
churn_flag
0    325
1    175
Name: count, dtype: int64
churn_flag
0    0.65
1    0.35
Name: proportion, dtype: float64


In [14]:
# Drop columns 

cols_to_drop = [
    # identifiers
    'account_id', 'account_name',

    # leakage — recorded AFTER churn happens
    'churn_event_count', 'total_refund',
    'ever_reactivated', 'most_recent_reason',

    # duplicates / redundant
    'plan_tier',            # duplicate of plan_tier_sub
    'seats',                # duplicate of total_seats_sub
    'mean_mrr',             # correlated with max_mrr
    'mean_first_response',  # correlated with mean_resolution_hrs
    'total_usage_duration', # correlated with total_usage_count
]

# only drop cols that actually exist (safety check)
cols_to_drop = [c for c in cols_to_drop if c in df.columns]
df.drop(columns=cols_to_drop, inplace=True)

print("Shape after drops:", df.shape)
print("Remaining columns:\n", df.columns.tolist())

Shape after drops: (500, 25)
Remaining columns:
 ['industry', 'country', 'signup_date', 'referral_source', 'is_trial', 'churn_flag', 'max_mrr', 'total_seats_sub', 'sub_count', 'ever_upgraded', 'ever_downgraded', 'auto_renew', 'billing_frequency', 'plan_tier_sub', 'ticket_count', 'mean_resolution_hrs', 'mean_satisfaction', 'escalation_count', 'total_usage_count', 'total_errors', 'distinct_features', 'beta_feature_usage', 'preceding_downgrade', 'preceding_upgrade', 'has_satisfaction_score']


In [15]:
#  Fix dtypes 

# boolean columns — cast to int (0/1) so sklearn can handle them
bool_cols = ['is_trial', 'churn_flag', 'ever_upgraded', 'ever_downgraded',
             'auto_renew', 'preceding_downgrade', 'preceding_upgrade',
             'has_satisfaction_score']

bool_cols = [c for c in bool_cols if c in df.columns]

for col in bool_cols:
    df[col] = df[col].astype(str).str.strip().str.lower()
    df[col] = df[col].map({'true': 1, 'false': 0, '1': 1, '0': 0, 
                           'yes': 1, 'no': 0}).fillna(0).astype(int)

# numeric columns — force to float
num_cols = ['max_mrr', 'sub_count', 'ticket_count', 'mean_resolution_hrs',
            'mean_satisfaction', 'escalation_count', 'total_usage_count',
            'total_errors', 'distinct_features', 'beta_feature_usage',
            'tenure_days']

num_cols = [c for c in num_cols if c in df.columns]

for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

# Transform signup_date → tenure_days (safe: signup happened before churn)
df['signup_date']  = pd.to_datetime(df['signup_date'])
df['tenure_days']  = (pd.Timestamp.today() - df['signup_date']).dt.days
df.drop(columns=['signup_date'], inplace=True)

print("Dtypes after fix:")
print(df.dtypes)
print("\nNulls after dtype fix:", df.isnull().sum().sum())

Dtypes after fix:
industry                      str
country                       str
referral_source               str
is_trial                    int64
churn_flag                  int64
max_mrr                     int64
total_seats_sub             int64
sub_count                   int64
ever_upgraded               int64
ever_downgraded             int64
auto_renew                  int64
billing_frequency             str
plan_tier_sub                 str
ticket_count              float64
mean_resolution_hrs       float64
mean_satisfaction         float64
escalation_count          float64
total_usage_count           int64
total_errors                int64
distinct_features           int64
beta_feature_usage          int64
preceding_downgrade         int64
preceding_upgrade           int64
has_satisfaction_score      int64
tenure_days                 int64
dtype: object

Nulls after dtype fix: 0


In [16]:
# Encode categoricals 

#  Label encode binary / ordinal categoricals 

# These have 2 values or a natural order (e.g. plan tiers)

label_enc_cols = ['billing_frequency',   # monthly / annual
                  'plan_tier_sub',        # starter / pro / enterprise (ordinal)
                  'company_size']         # Small / Mid / Large / Enterprise

label_enc_cols = [c for c in label_enc_cols if c in df.columns]

label_encoders = {}   # save each encoder — needed later in Streamlit app
for col in label_enc_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le
    print(f"{col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# One-hot encode nominal categoricals 

# These have no order — industry, region, referral_source

ohe_cols = ['industry', 'region', 'referral_source','country']
ohe_cols = [c for c in ohe_cols if c in df.columns]

df = pd.get_dummies(df, columns=ohe_cols, drop_first=True)

print("\nShape after encoding:", df.shape)
print("New OHE columns:", [c for c in df.columns if any(c.startswith(o) for o in ohe_cols)])

billing_frequency: {'annual': np.int64(0), 'monthly': np.int64(1)}
plan_tier_sub: {'Basic': np.int64(0), 'Enterprise': np.int64(1), 'Pro': np.int64(2)}

Shape after encoding: (500, 36)
New OHE columns: ['industry_DevTools', 'industry_EdTech', 'industry_FinTech', 'industry_HealthTech', 'referral_source_event', 'referral_source_organic', 'referral_source_other', 'referral_source_partner', 'country_CA', 'country_DE', 'country_FR', 'country_IN', 'country_UK', 'country_US']


In [17]:
# Feature engineering 

# 1 Error rate - errors per usage session (frustration ratio)
df['error_rate'] = np.where(
    df['total_usage_count'] > 0,
    df['total_errors'] / df['total_usage_count'],
    0
)

# 2 Ticket rate - support burden relative to tenure
df['ticket_rate'] = np.where(
    df['tenure_days'] > 0,
    df['ticket_count'] / (df['tenure_days'] / 30),  # tickets per month
    0
)

# 3 MRR per seat - revenue intensity
df['mrr_per_seat'] = np.where(
    df['total_seats_sub'] > 0 if 'total_seats_sub' in df.columns
    else df['max_mrr'] > 0,
    df['max_mrr'] / df.get('total_seats_sub', df['max_mrr']),
    0
)

# 4 Feature adoption score - how broadly they use the product (0 to 1)
max_features = df['distinct_features'].max()
df['feature_adoption_score'] = df['distinct_features'] / max_features

# 5 Is low usage flag - bottom 25% of usage
usage_25th = df['total_usage_count'].quantile(0.25)
df['is_low_usage'] = (df['total_usage_count'] <= usage_25th).astype(int)

# 6 Tenure group - bucketed (useful for dashboard too)
df['tenure_group'] = pd.cut(
    df['tenure_days'],
    bins=[0, 90, 180, 365, 730, 99999],
    labels=['0-3mo', '3-6mo', '6-12mo', '1-2yr', '2yr+']
)
# encode it immediately
df['tenure_group'] = LabelEncoder().fit_transform(df['tenure_group'].astype(str))

print("New engineered features:")
new_feats = ['error_rate', 'ticket_rate', 'mrr_per_seat',
             'feature_adoption_score', 'is_low_usage', 'tenure_group']
print(df[new_feats].describe().round(3))

New engineered features:
       error_rate  ticket_rate  mrr_per_seat  feature_adoption_score  \
count     500.000      500.000       500.000                 500.000   
mean        0.056        0.159       134.412                   0.603   
std         0.016        0.091        62.176                   0.111   
min         0.000        0.000         5.429                   0.308   
25%         0.046        0.092        79.187                   0.538   
50%         0.055        0.143       141.310                   0.615   
75%         0.066        0.207       199.000                   0.692   
max         0.110        0.525       199.000                   1.000   

       is_low_usage  tenure_group  
count       500.000       500.000  
mean          0.252         0.590  
std           0.435         0.492  
min           0.000         0.000  
25%           0.000         0.000  
50%           0.000         1.000  
75%           1.000         1.000  
max           1.000         1.000  


In [18]:
#  X / y split + train/test split 

# separate target from features
y = df['churn_flag'].astype(int)
X = df.drop(columns=['churn_flag'])

print("X shape:", X.shape)
print("y distribution:\n", y.value_counts())

# train / test split — stratify keeps class ratio equal in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,          # 80% train, 20% test
    random_state=42,        # reproducibility
    stratify=y              # IMPORTANT: maintains churn ratio in both splits
)

print("\nTrain size:", X_train.shape)
print("Test size:", X_test.shape)
print("\nTrain churn rate:", y_train.mean().round(3))
print("Test churn rate: ", y_test.mean().round(3))  # should match train closely

X shape: (500, 41)
y distribution:
 churn_flag
0    325
1    175
Name: count, dtype: int64

Train size: (400, 41)
Test size: (100, 41)

Train churn rate: 0.35
Test churn rate:  0.35


In [19]:
# Scale numerical features 

# identify numeric columns to scale
# exclude: already-binary cols (0/1), OHE cols, engineered flags
do_not_scale = ['churn_flag', 'is_trial', 'ever_upgraded', 'ever_downgraded',
                'auto_renew', 'preceding_downgrade', 'preceding_upgrade',
                'has_satisfaction_score', 'is_low_usage']

# get all numeric cols not in the exclusion list
scale_cols = [c for c in X_train.select_dtypes(include='number').columns
              if c not in do_not_scale
              and not any(c.startswith(ohe) for ohe in
                          ['industry_', 'region_', 'referral_source_'])]

print("Columns being scaled:", scale_cols)

# fit scaler on TRAIN only
scaler = StandardScaler()
X_train[scale_cols] = scaler.fit_transform(X_train[scale_cols])

# transform TEST using the SAME fitted scaler
X_test[scale_cols]  = scaler.transform(X_test[scale_cols])

print("\nX_train after scaling (sample):")
print(X_train[scale_cols].describe().round(2))

Columns being scaled: ['max_mrr', 'total_seats_sub', 'sub_count', 'billing_frequency', 'plan_tier_sub', 'ticket_count', 'mean_resolution_hrs', 'mean_satisfaction', 'escalation_count', 'total_usage_count', 'total_errors', 'distinct_features', 'beta_feature_usage', 'tenure_days', 'error_rate', 'ticket_rate', 'mrr_per_seat', 'feature_adoption_score', 'tenure_group']

X_train after scaling (sample):
       max_mrr  total_seats_sub  sub_count  billing_frequency  plan_tier_sub  \
count   400.00           400.00     400.00             400.00         400.00   
mean      0.00             0.00      -0.00               0.00          -0.00   
std       1.00             1.00       1.00               1.00           1.00   
min      -1.51            -1.88      -2.45              -1.04          -1.20   
25%      -0.67            -0.69      -0.91              -1.04          -1.20   
50%      -0.24            -0.18       0.02               0.96           0.05   
75%       0.44             0.47       0.6

In [20]:
# Final checks + save all outputs 

#  sanity checks

print("=== Final Sanity Checks ===")
print("X_train shape:", X_train.shape)
print("X_test shape: ", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape: ", y_test.shape)
print("Any nulls in X_train:", X_train.isnull().sum().sum())
print("Any nulls in X_test: ", X_test.isnull().sum().sum())
print("Train churn rate:", y_train.mean().round(3))
print("Test churn rate: ", y_test.mean().round(3))
print("Feature count:", X_train.shape[1])

# save processed files 
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../models', exist_ok=True)

X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv',  index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv',  index=False)

# save the full preprocessed df (needed for dashboard export later)
df.to_csv('../data/processed/df_preprocessed.csv', index=False)

# save scaler and encoders — CRITICAL for Streamlit app 
joblib.dump(scaler,         '../models/scaler.pkl')
joblib.dump(label_encoders, '../models/label_encoders.pkl')

print("\n=== Saved ===")
print("X_train.csv, X_test.csv, y_train.csv, y_test.csv")
print("df_preprocessed.csv")
print("scaler.pkl")
print("label_encoders.pkl")

=== Final Sanity Checks ===
X_train shape: (400, 41)
X_test shape:  (100, 41)
y_train shape: (400,)
y_test shape:  (100,)
Any nulls in X_train: 0
Any nulls in X_test:  0
Train churn rate: 0.35
Test churn rate:  0.35
Feature count: 41

=== Saved ===
X_train.csv, X_test.csv, y_train.csv, y_test.csv
df_preprocessed.csv
scaler.pkl
label_encoders.pkl
